In [2]:
import pandas as pd
import os
import logging
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import nltk

In [3]:
#Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

In [4]:
#Ensure VADER is available
try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    logging.info("Downloading VADER sentiment analysis tool...")
    nltk.download("vader_lexicon")

In [5]:
#Define file paths
final_dir = os.path.join("..", "data", "final")
sentiment_dir = os.path.join("..", "data", "sentiment")
os.makedirs(sentiment_dir, exist_ok=True)

input_file = os.path.join(final_dir, "final_cleaned_data.csv")
output_file = os.path.join(sentiment_dir, "sentiment_analysis.csv")

In [6]:
#Load dataset
logging.info(f"Loading final cleaned dataset from: {input_file}")
df = pd.read_csv(input_file)

2025-03-23 10:26:43,748 - INFO - Loading final cleaned dataset from: ../data/final/final_cleaned_data.csv


In [7]:
#Ensure required columns exist
if "text" not in df.columns or "Source" not in df.columns:
    raise ValueError("Missing required columns ('text' and 'Source') in final dataset!")


In [8]:
#Initialize VADER
sia = SentimentIntensityAnalyzer()

In [9]:
#Custom Political Sentiment Keywords
negative_keywords = ["election", "wasn’t elected", "demand election", "not democratic", "Carney not elected"]
positive_keywords = ["support Carney", "good choice", "experience", "qualified", "capable"]

In [10]:
#Function to get sentiment scores
def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0, 0, "Neutral", "No"  # Default values

    #VADER Sentiment Score
    vader_score = sia.polarity_scores(text)["compound"]

    #TextBlob Sentiment Score
    blob_score = TextBlob(text).sentiment.polarity

    #Weighted Average (VADER = 70%, TextBlob = 30%)
    final_score = (0.7 * vader_score) + (0.3 * blob_score)

    #Sentiment Thresholds (Narrowed Neutral Range)
    if final_score > 0.2:
        sentiment = "Positive"
    elif final_score < -0.2:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    #Override Classification Based on Custom Keywords
    text_lower = text.lower()
    if any(word in text_lower for word in negative_keywords):
        sentiment = "Negative"
    elif any(word in text_lower for word in positive_keywords):
        sentiment = "Positive"

    #Detect Strong Disagreement
    disagreement_flag = "Yes" if (vader_score * blob_score < 0) else "No"

    return vader_score, blob_score, sentiment, disagreement_flag

In [11]:
#Apply sentiment analysis
logging.info(f"Applying sentiment analysis to {len(df)} records.")
df[["vader_score", "textblob_score", "final_sentiment", "disagreement_flag"]] = df["text"].apply(
    lambda x: pd.Series(get_sentiment(x))
)

2025-03-23 10:29:06,491 - INFO - Applying sentiment analysis to 1507 records.


In [12]:
#Save results
df.to_csv(output_file, index=False)
logging.info(f"✅ Sentiment analysis complete! Results saved to: {output_file}")

2025-03-23 10:29:18,841 - INFO - ✅ Sentiment analysis complete! Results saved to: ../data/sentiment/sentiment_analysis.csv
